In [2]:
# Initialize Otter
import otter
from numpy.ma.core import ones_like

grader = otter.Notebook("DS3000-9000_A02.ipynb")

# Assignment 2: Linear Regression

**DS 3000A/9000A Fall 2026**

This assignment will be graded on Gradescope.

1. Run `pip install otter-grader` first, if you don't have it installed.
2. Included in this notebook are public test cases, which you can use to verify your answers before submitting.
2. Additional private test cases will run on Gradescope upon submission. You will see your score but not the tests themselves.
3. You can revise and re-upload your submission on Gradescope up until the deadline.

In [3]:
## Install packages if neccessary.
## Uncomment and run if you need to.

# !pip install numpy pandas matplotlib

In [4]:
import numpy as np
import pandas as pd

---

## Question 1
 
**50 Points**
 
In the lab, the input $x \in \mathbb{R}$ was a scalar (year). In practice, inputs are multidimensional: $\mathbf{x} \in \mathbb{R}^D$.

To capture nonlinear relationships, expand each feature independently up to degree $d$ and prepend a bias:
 
$$
\boldsymbol{\Phi}(\mathbf{X}, d) = \bigl[\mathbf{1},\ \mathbf{X}^{\odot 1},\ \mathbf{X}^{\odot 2},\ \ldots,\ \mathbf{X}^{\odot d}\bigr] \in \mathbb{R}^{N \times (D d + 1)}
$$
 
where $\mathbf{X}^{\odot p}$ is elementwise power. The normal equation applies with $\boldsymbol{\Phi}$ replacing $\hat{\mathbf{X}}$.
 
**Setup.** We predict **Canada's GDP** using other G7 countries' GDPs as features.

In [5]:
df = pd.read_csv("owid-gdp.csv")
df["GDP"] = df["GDP"] / 1e12

FEATURES = ["United States", "United Kingdom", "Germany", "France", "Japan", "Italy"]
TARGET = "Canada"

gdp = pd.DataFrame(
    {
        entity: df[df["Entity"] == entity].set_index("Year")["GDP"]
        for entity in FEATURES + [TARGET]
    }
).dropna()

train = gdp[gdp.index < 2000]
test = gdp[gdp.index >= 2000]

X_train = train[FEATURES].values
y_train = train[TARGET].values
X_test = test[FEATURES].values
y_test = test[TARGET].values

print("X_train shape:", X_train.shape, "y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape, "y_test shape:", y_test.shape)

X_train shape: (40, 6) y_train shape: (40,)
X_test shape: (25, 6) y_test shape: (25,)


1. Implement `phi(X, degree)` returning $\boldsymbol{\Phi}(\mathbf{X}, d)$ of shape $(N,\ D d + 1)$.

2. Implement `fit_phi(X, y, degree)` that fits $\boldsymbol{\theta}$ using `phi`.

3. Similarly, implement `predict_phi`.

4. For each degree in `degrees = [1, 2, 3, 5]`, fit a model on the training set and compute train and test RMSE. Store the results as a DataFrame `rmse_df` with columns `degree`, `train_rmse`, `test_rmse`.

**HINT:** If `np.linalg.inv` throws the exception `LinAlgError: Singular matrix`, add small values to the diagonal of the matrix you're trying to inverse, e.g., `M + 0.001 * np.eye(M.shape[0])`.

In [6]:
def phi(X: np.ndarray, degree: int) -> np.ndarray:
    """Compute polynomial feature matrix.

    Args:
        X: Input matrix of shape (N, D).
        degree: Polynomial degree d >= 1.

    Returns:
        Feature matrix of shape (N, D*degree + 1).
        Columns: [1, X, X^2, ..., X^d] (elementwise powers).
    """
    ones = np.ones((X.shape[0], 1))
    for i in range(1, degree+1):
        ones = np.hstack((ones, X**i))
    return ones


def fit_phi(X: np.ndarray, y: np.ndarray, degree: int) -> np.ndarray:
    """Fit linear regression with polynomial basis functions.

    Args:
        X: Input matrix of shape (N, D).
        y: Target array of shape (N,).
        degree: Polynomial degree.

    Returns:
        Parameter vector theta of shape (D*degree + 1,).
    """
    Phi = phi(X, degree)
    return np.linalg.inv(Phi.T @ Phi) @ Phi.T @ y


def predict_phi(X: np.ndarray, theta: np.ndarray, degree: int) -> np.ndarray:
    """Predict using polynomial features and parameters.

    Args:
        X: Input matrix of shape (N, D).
        theta: Parameter vector of shape (D*degree + 1,).
        degree: Polynomial degree.

    Returns:
        Predicted values of shape (N,).
    """
    return phi(X, degree) @ theta


def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Compute root mean squared error.

    Args:
        y_true: True target values of shape (N,).
        y_pred: Predicted values of shape (N,).

    Returns:
        RMSE as a float.
    """
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


degrees = [1, 2, 3, 5]

train, test = [], []

for degree in degrees:
    train.append(rmse(y_train, predict_phi(X_train, fit_phi(X_train, y_train, degree), degree)))
    test.append(rmse(y_test, predict_phi(X_test, fit_phi(X_test, y_test, degree), degree)))

rmse_df = pd.DataFrame({"degree": degrees, "train_rmse": train, "test_rmse": test})

rmse_df

,degree,train_rmse,test_rmse
0,1,0.006894,0.007962
1,2,0.005320,0.005970
2,3,0.004592,0.005025
3,5,0.001608,3.360651


In [7]:
grader.check("q1")

q1 results: All test cases passed!

---

## Question 2
 
**50 Points**
 
So far, we predicted a single target $y \in \mathbb{R}$. In **multi-output regression**, we predict $K$ targets simultaneously. Stack the targets into a matrix $\mathbf{Y} \in \mathbb{R}^{N \times K}$; the normal equation extends naturally: (Assuming $\Phi \in \mathbb{R}^{N \times B}$ for some $B$, e.g., $Dd + 1$ in the previous question.)
 
$$
\boldsymbol{\Theta} = (\boldsymbol{\Phi}^\top \boldsymbol{\Phi})^{-1} \boldsymbol{\Phi}^\top \mathbf{Y} \in \mathbb{R}^{B \times K}
$$
 
Each column of $\boldsymbol{\Theta}$ is an independent set of parameters for one output. Predictions become:
 
$$
\hat{\mathbf{Y}} = \boldsymbol{\Phi} \boldsymbol{\Theta} \in \mathbb{R}^{N \times K}
$$
 
**Setup.** We now predict **all non-G7 anglophone countries' GDPs** simultaneously, using the G7 countries as features.

In [8]:
df = pd.read_csv("owid-gdp.csv")
df["GDP"] = df["GDP"] / 1e12

FEATURES = [
    "United States",
    "United Kingdom",
    "Germany",
    "France",
    "Japan",
    "Italy",
    "Canada",
]
TARGETS = ["Australia", "New Zealand", "Ireland"]

gdp = pd.DataFrame(
    {
        entity: df[df["Entity"] == entity].set_index("Year")["GDP"]
        for entity in FEATURES + TARGETS
    }
).dropna()

train = gdp[gdp.index < 2000]
test = gdp[gdp.index >= 2000]

X_train = train[FEATURES].values
Y_train = train[TARGETS].values
X_test = test[FEATURES].values
Y_test = test[TARGETS].values

print("X_train shape:", X_train.shape, "y_train shape:", Y_train.shape)
print("X_test shape:", X_test.shape, "y_test shape:", Y_test.shape)

X_train shape: (40, 7) y_train shape: (40, 3)
X_test shape: (25, 7) y_test shape: (25, 3)


1. Implement `fit_multi(X, Y)` that computes $\boldsymbol{\Theta} \in \mathbb{R}^{(Dd+1) \times K}$ via the normal equation. Use the polynomial basis function with degree 2.

2. Implement `predict_multi(X, Theta)` that returns $\hat{\mathbf{Y}} \in \mathbb{R}^{N \times K}$.

3. Fit the model and assign results to `Theta`. Compute per-target test RMSE and store as a `pd.Series` named `rmse_per_target` indexed by target country name.

In [9]:
def fit_multi(X: np.ndarray, Y: np.ndarray) -> np.ndarray:
    """Fit multi-output linear regression via the normal equation with
    degree-2 polynomial basis function.

    Args:
        X: Input matrix of shape (N, D).
        Y: Target matrix of shape (N, K).

    Returns:
        Parameter matrix Theta of shape (2D+1, K).
    """
    Phi = phi(X, degree=2)
    return np.linalg.inv(Phi.T @ Phi) @ Phi.T @ Y


def predict_multi(X: np.ndarray, Theta: np.ndarray) -> np.ndarray:
    """Predict multiple outputs with degree-2 polynomial basis function

    Args:
        X: Input matrix of shape (N, D).
        Theta: Parameter matrix of shape (2D+1, K).

    Returns:
        Predicted output matrix of shape (N, K).
    """
    return phi(X, 2) @ Theta



Theta = fit_multi(X_train, Y_train)

Y_pred_test = predict_multi(X_test, Theta)

data = []
for c in range(len(TARGETS)):
    data.append(rmse(Y_test[:, c], Y_pred_test[:, c]))
rmse_per_target = pd.Series(data, index=TARGETS)
rmse_per_target

Australia      0.164482
New Zealand    0.031452
Ireland        0.053565
dtype: float64

In [10]:
grader.check("q2")

q2 results: All test cases passed!

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [11]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False, run_tests=True)

Running your submission against local test cases...


Your submission received the following results when run against available test cases:

    q1 results: All test cases passed!

    q2 results: All test cases passed!


C:\Users\Owner\AppData\Local\Programs\Python\Python313\Lib\site-packages\otter\check\utils.py:115: RuntimeWarning: [IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.

  warnings.warn(results.stderr.decode("utf-8"), RuntimeWarning)
